## UTS Praktikum Visi Komputer B
## Pemindai Dokumen dan Nota Otomatis (CamScanner Mini)

#### Kelompok 1 
``` Anggota ```
 * Muhammad Hafidz 2408107010052
 * Teuku Fikram Al Syahbanna 24081070100
 * Ahmad Hanif 2408107010114

Program ini mengubah foto dokumen dari HP (miring, ada background meja, cahaya tidak rata) menjadi gambar yang tampak seperti hasil scan: kertas lurus, memenuhi gambar, dan tulisan jelas.

Ada dua mode:

    * Mode A: satu halaman kertas yang terlihat penuh dalam satu foto.
    * Mode B: dokumen lebar yang difoto dua kali (bagian kiri dan kanan saling tumpang tindih), lalu disambung dengan ORB dan homografi.

Catatan penggunaan:
``` 
    1) Ganti Semua URL pada bagian Konfigurasi dengan alamat foto kalian yang sudah    diunggah ke website.

    2) Semua gambar ditampilkan dengan matplotlib (bukan cv2.imshow) supaya muncul di file html.

    3) Jalankan seluruh cell dari atas ke bawah sebelum mengekspor html.
```

### Persiapan Library


In [1]:
%pip install -q opencv-python-headless requests pandas matplotlib numpy

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.0.1 -> 26.2.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import time

import cv2
import numpy as np
import pandas as pd
import requests
import matplotlib.pyplot as plt

print("OpenCV versi:", cv2.__version__)

OpenCV versi: 5.0.0


### Konfigurasi
Semua gambar dibaca dari URL, tidak disimpan di komputer.

    * a4 diisi True jika kertas yang difoto berukuran A4 (dipakai untuk menghitung selisih rasio saat evaluasi).
    * kondisi hanya label untuk tabel evaluasi, misalnya terang, redup, bayangan, miring, background gelap.

In [ ]:
# Parameter pemrosesan
TINGGI_PROSES = 500      # tinggi gambar saat mencari tepi (piksel)
LEBAR_HASIL = 1200       # lebar maksimum hasil scan (gambar kecil tidak diperbesar)
MARGIN_POTONG = 0.015    # sisa tepi meja dibuang 1,5% dari tiap sisi hasil
MIN_LUAS_DOKUMEN = 0.2   # kontur dokumen minimal 20% luas gambar
MAKS_LUAS_DOKUMEN = 0.95 # kontur lebih dari 95% luas gambar dianggap bingkai foto
PEMBAGI_BLOK = 40        # ukuran blok adaptive threshold = lebar hasil / 40
KONSTANTA_C = 10         # pengurang pada adaptive threshold
RASIO_A4 = 297 / 210     # tinggi dibagi lebar kertas A4

# Folder foto di GitHub (repository harus public)
BASE_URL = "https://raw.githubusercontent.com/Nixzouxu/CamScanner-Mini/main/Gambar/"


def url_foto(nama_file):
    # Menyusun URL foto dan mengubah spasi pada nama file menjadi %20
    return BASE_URL + quote(nama_file)


# Foto uji Mode A
DAFTAR_FOTO = [
    {"nama": "Foto 1", "kondisi": "terang, miring sedang",   "a4": True,  "url": url_foto("Foto1.jpeg")},
    {"nama": "Foto 2", "kondisi": "redup",                   "a4": True,  "url": url_foto("Foto 2.jpeg")},
    {"nama": "Foto 3", "kondisi": "bayangan",                "a4": True,  "url": url_foto("Foto 3.jpeg")},
    {"nama": "Foto 4", "kondisi": "miring ekstrem",          "a4": True,  "url": url_foto("Foto 4.jpeg")},
    {"nama": "Foto 5", "kondisi": "background hitam",        "a4": True,  "url": url_foto("Foto 5.jpeg")},
    {"nama": "Foto 6", "kondisi": "kontras rendah (putih)",  "a4": True,  "url": url_foto("Foto 6.jpeg")},
    {"nama": "Foto 7", "kondisi": "struk (nota)",            "a4": False, "url": url_foto("Foto 7.jpeg")},
]

# Foto lebar Mode B: satu foto dipotong di program menjadi bagian kiri dan kanan
# yang saling tumpang tindih (kiri: 0 sampai 62% lebar, kanan: 38 sampai 100% lebar)
FOTO_LEBAR = [
    {"nama": "buku_terbuka", "url": url_foto("Foto 8.jpeg"), "batas_kiri": 0.62, "batas_kanan": 0.38},
]

### Fungsi Utilitas
Tiga fungsi dasar: membaca gambar dari URL, menampilkan gambar, dan mengubah lebar gambar.

cv2.imread tidak bisa membaca alamat web. Karena itu byte gambar diambil dengan requests, lalu didecode dengan cv2.imdecode.

In [ ]:
def baca_gambar_url(url):
    # Mengunduh gambar dari URL dan mengubahnya menjadi array BGR OpenCV
    respon = requests.get(url, timeout=30, headers={"User-Agent": "Mozilla/5.0"})
    respon.raise_for_status()

    data = np.frombuffer(respon.content, dtype=np.uint8)
    gambar = cv2.imdecode(data, cv2.IMREAD_COLOR)
    if gambar is None:
        raise ValueError(f"Isi URL bukan gambar yang valid: {url}")
    return gambar


def tampilkan(daftar_gambar, daftar_judul, kolom=3, ukuran=5):
    # Menampilkan beberapa gambar sekaligus (BGR atau grayscale) dengan matplotlib
    baris = int(np.ceil(len(daftar_gambar) / kolom))
    plt.figure(figsize=(ukuran * kolom, ukuran * baris))

    for i, (gambar, judul) in enumerate(zip(daftar_gambar, daftar_judul), start=1):
        plt.subplot(baris, kolom, i)
        if gambar.ndim == 3:
            plt.imshow(cv2.cvtColor(gambar, cv2.COLOR_BGR2RGB))
        else:
            plt.imshow(gambar, cmap="gray", vmin=0, vmax=255)
        plt.title(judul)
        plt.axis("off")

    plt.tight_layout()
    plt.show()


def ubah_lebar(gambar, lebar_baru):
    # Mengubah lebar gambar ke lebar_baru dengan rasio tetap
    skala = lebar_baru / gambar.shape[1]
    tinggi_baru = int(gambar.shape[0] * skala)
    return cv2.resize(gambar, (lebar_baru, tinggi_baru), interpolation=cv2.INTER_AREA)

### Histogram
Histogram dihitung manual dengan nested loop seperti pada Prak 1, lalu divalidasi dengan cv2.calcHist. Fungsi ini dipakai nanti untuk membandingkan citra sebelum dan sesudah equalization.

In [ ]:
def histogram_manual(abu):
    # Menghitung frekuensi tiap intensitas 0-255 dengan nested loop
    hist = np.zeros(256, dtype=int)
    for baris in range(abu.shape[0]):
        for kolom in range(abu.shape[1]):
            hist[abu[baris, kolom]] += 1
    return hist


def validasi_histogram(abu):
    # Memastikan histogram manual sama dengan hasil cv2.calcHist
    manual = histogram_manual(abu)
    bawaan = cv2.calcHist([abu], [0], None, [256], [0, 256]).ravel().astype(int)
    return np.array_equal(manual, bawaan)

## Fungsi Inti Mode A

#### Preprocessing: perkecil, grayscale, blur, Canny

Gambar diperkecil supaya proses cepat dan tepinya lebih bersih. Rasio perkecilan disimpan agar sudut yang ditemukan bisa dikembalikan ke ukuran foto asli. Blur dipakai untuk meredam noise sebelum Canny (Prak 4). Hasil Canny dilebarkan sedikit dengan dilate agar garis tepi yang putus tersambung.

In [ ]:
def praproses(gambar, tinggi=TINGGI_PROSES):
    # Mengembalikan gambar kecil, peta tepi Canny, dan rasio perkecilan
    rasio = gambar.shape[0] / tinggi
    lebar = int(gambar.shape[1] / rasio)
    kecil = cv2.resize(gambar, (lebar, tinggi), interpolation=cv2.INTER_AREA)

    abu = cv2.cvtColor(kecil, cv2.COLOR_BGR2GRAY)
    blur = cv2.GaussianBlur(abu, (5, 5), 0)

    tepi = cv2.Canny(blur, 75, 200)
    tepi = cv2.dilate(tepi, np.ones((3, 3), np.uint8), iterations=1)
    return kecil, tepi, rasio

### Mencari 4 sudut kertas
Kertas biasanya objek terbesar di foto, jadi kontur diurutkan berdasarkan luas. Setiap kontur disederhanakan dengan approxPolyDP. Toleransinya dinaikkan bertahap (2%, 3%, 4% dari keliling) karena tepi kertas asli sering sedikit melengkung. Kontur yang hampir seluas seluruh gambar (lebih dari 95%) diabaikan karena itu biasanya bingkai foto, bukan kertas.

Kontur pertama yang menjadi 4 titik dan cukup besar dianggap kertas. Jika tidak ada, dipakai persegi miring terkecil (minAreaRect) sebagai cadangan dan hasilnya ditandai gagal.

In [ ]:
def cari_sudut_dokumen(tepi):
    # Mengembalikan (4 titik sudut, status berhasil)
    # Status False berarti memakai cadangan
    kontur, _ = cv2.findContours(tepi.copy(), cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    if len(kontur) == 0:
        return None, False

    kontur = sorted(kontur, key=cv2.contourArea, reverse=True)[:5]
    luas_gambar = tepi.shape[0] * tepi.shape[1]
    luas_min = MIN_LUAS_DOKUMEN * luas_gambar
    luas_maks = MAKS_LUAS_DOKUMEN * luas_gambar

    for c in kontur:
        if not (luas_min <= cv2.contourArea(c) <= luas_maks):
            continue
        keliling = cv2.arcLength(c, True)
        for toleransi in (0.02, 0.03, 0.04):
            hampiran = cv2.approxPolyDP(c, toleransi * keliling, True)
            if len(hampiran) == 4:
                return hampiran.reshape(4, 2).astype("float32"), True

    # Cadangan: persegi miring terkecil dari kontur terbesar
    kotak = cv2.boxPoints(cv2.minAreaRect(kontur[0]))
    return kotak.astype("float32"), False

### Mengurutkan sudut dan meluruskan perspektif

1) Mengurutkan sudut. 
    Titik dari kontur urutannya acak, sedangkan transformasi butuh urutan tetap: kiri atas, kanan atas, kanan bawah, kiri bawah. Cara sederhana (jumlah x+y dan selisih y-x) gagal jika kertas terputar lebih dari sekitar 45 derajat, karena satu sudut bisa terpilih dua kali. Karena itu titik diurutkan berdasarkan sudut terhadap titik pusat, lalu dimulai dari titik paling kiri atas.

2) Menentukan ukuran hasil. 
    Lebar diambil dari sisi atas dan bawah kertas. Tingginya tidak diambil dari panjang sisi di foto, karena pada foto yang miring sisi tegak kertas tampak memendek dan hasilnya jadi gepeng. Sebagai gantinya, rasio lebar dan tinggi kertas ditaksir dari bentuk keempat sudut memakai model kamera sederhana (pusat gambar dianggap pusat lensa dan piksel dianggap persegi). Metode ini dikenal dari penelitian pemindaian papan tulis oleh Zhang dan He (2007). Jika taksirannya tidak masuk akal, program memakai panjang sisi biasa.

3) Meluruskan. 
    getPerspectiveTransform dan warpPerspective menarik keempat sudut menjadi persegi panjang. Secara konsep ini sama dengan homografi pada Prak 5, hanya pasangan titiknya berasal dari 4 sudut kertas.

In [ ]:
def urutkan_titik(titik):
    # Mengurutkan 4 titik searah jarum jam mulai dari kiri atas (tahan terhadap rotasi)
    pusat = titik.mean(axis=0)
    sudut = np.arctan2(titik[:, 1] - pusat[1], titik[:, 0] - pusat[0])
    urut = titik[np.argsort(sudut)]
    mulai = np.argmin(urut.sum(axis=1))
    return np.roll(urut, -mulai, axis=0).astype("float32")


def taksir_rasio(sudut, ukuran_gambar):
    # Menaksir rasio lebar/tinggi kertas asli dari 4 sudut yang terkena perspektif
    # Mengembalikan None jika perhitungan tidak masuk akal
    tinggi_g, lebar_g = ukuran_gambar
    pusat = np.array([lebar_g / 2, tinggi_g / 2])

    # Koordinat homogen relatif terhadap pusat gambar
    m1, m2, m4, m3 = [np.append(p - pusat, 1.0) for p in sudut]

    pembagi2 = np.dot(np.cross(m2, m4), m3)
    pembagi3 = np.dot(np.cross(m3, m4), m2)
    if abs(pembagi2) < 1e-9 or abs(pembagi3) < 1e-9:
        return None

    k2 = np.dot(np.cross(m1, m4), m3) / pembagi2
    k3 = np.dot(np.cross(m1, m4), m2) / pembagi3
    n2 = k2 * m2 - m1     # arah lebar kertas
    n3 = k3 * m3 - m1     # arah tinggi kertas

    penyebut = n2[2] * n3[2]
    if abs(penyebut) < 1e-9:      # sisi kertas sejajar: tidak ada informasi perspektif
        return None
    f_kuadrat = -(n2[0] * n3[0] + n2[1] * n3[1]) / penyebut   # taksiran panjang fokus
    if f_kuadrat <= 0:
        return None

    panjang = lambda n: (n[0] ** 2 + n[1] ** 2) / f_kuadrat + n[2] ** 2
    rasio = float(np.sqrt(panjang(n2) / panjang(n3)))
    return rasio if 0.2 < rasio < 5 else None


def luruskan_perspektif(gambar, titik):
    # Mengubah area kertas yang miring menjadi tampak dari atas
    sudut = urutkan_titik(titik)
    kiri_atas, kanan_atas, kanan_bawah, kiri_bawah = sudut

    lebar = int(max(np.linalg.norm(kanan_atas - kiri_atas),
                    np.linalg.norm(kanan_bawah - kiri_bawah)))

    rasio = taksir_rasio(sudut, gambar.shape[:2])       # lebar / tinggi
    if rasio:
        tinggi = int(lebar / rasio)
    else:                                               # cadangan: panjang sisi di foto
        tinggi = int(max(np.linalg.norm(kiri_bawah - kiri_atas),
                         np.linalg.norm(kanan_bawah - kanan_atas)))

    tujuan = np.float32([[0, 0], [lebar - 1, 0],
                         [lebar - 1, tinggi - 1], [0, tinggi - 1]])
    matriks = cv2.getPerspectiveTransform(sudut, tujuan)
    return cv2.warpPerspective(gambar, matriks, (lebar, tinggi))

### Mempercantik hasil: tiga metode threshold
Tiga metode dibuat sekaligus untuk dibandingkan di bagian evaluasi:

* Global: satu nilai T untuk seluruh gambar, mudah gagal jika pencahayaan tidak rata.
* Otsu: nilai T dicari otomatis dari histogram, tetap satu nilai untuk seluruh gambar.
* Adaptive: nilai T dihitung per kawasan lokal, paling tahan terhadap bayangan.

In [ ]:
def percantik(gambar_bgr):
    # Menghasilkan tiga versi biner dari gambar yang sudah lurus
    abu = cv2.cvtColor(gambar_bgr, cv2.COLOR_BGR2GRAY)
    abu = cv2.GaussianBlur(abu, (5, 5), 0)

    # Ukuran blok mengikuti lebar gambar dan harus ganjil
    blok = max(11, abu.shape[1] // PEMBAGI_BLOK)
    if blok % 2 == 0:
        blok += 1

    _, global_ = cv2.threshold(abu, 127, 255, cv2.THRESH_BINARY)
    nilai_otsu, otsu = cv2.threshold(abu, 0, 255, cv2.THRESH_BINARY | cv2.THRESH_OTSU)
    adaptif = cv2.adaptiveThreshold(abu, 255, cv2.ADAPTIVE_THRESH_GAUSSIAN_C,
                                    cv2.THRESH_BINARY, blok, KONSTANTA_C)

    return {"abu": abu, "global": global_, "otsu": otsu,
            "adaptif": adaptif, "nilai_otsu": nilai_otsu}

### Fungsi gabungan: seluruh alur Mode A

In [ ]:
def pindai_dokumen(gambar):
    # Menjalankan seluruh alur Mode A dan mengembalikan semua hasil antara dalam satu dictionary
    waktu_mulai = time.perf_counter()

    kecil, tepi, rasio = praproses(gambar)
    sudut, berhasil = cari_sudut_dokumen(tepi)
    if sudut is None:
        return {"berhasil": False, "kecil": kecil, "tepi": tepi, "lurus": None}

    # Kembalikan sudut ke ukuran foto asli lalu luruskan
    lurus = luruskan_perspektif(gambar, sudut * rasio)

    # Buang sisa tepi meja, lalu perkecil jika terlalu lebar (tidak pernah diperbesar)
    margin = int(MARGIN_POTONG * min(lurus.shape[:2]))
    lurus = lurus[margin:-margin, margin:-margin]
    if lurus.shape[1] > LEBAR_HASIL:
        lurus = ubah_lebar(lurus, LEBAR_HASIL)
    versi = percantik(lurus)

    # Gambar visualisasi hasil deteksi sudut (hijau = berhasil, merah = cadangan)
    visual = kecil.copy()
    warna = (0, 255, 0) if berhasil else (0, 0, 255)
    cv2.polylines(visual, [sudut.astype(np.int32)], True, warna, 3)

    return {"berhasil": berhasil, "kecil": kecil, "tepi": tepi, "visual": visual,
            "lurus": lurus, "versi": versi,
            "waktu": time.perf_counter() - waktu_mulai}

### Demonstrasi Langkah demi Langkah (Satu Foto)
Bagian ini menampilkan setiap tahap pada foto pertama supaya alurnya mudah dijelaskan di laporan dan video.

In [ ]:
foto_contoh = baca_gambar_url(DAFTAR_FOTO[0]["url"])
hasil = pindai_dokumen(foto_contoh)

print("Ukuran foto asli :", foto_contoh.shape)
print("Deteksi 4 sudut  :", "berhasil" if hasil["berhasil"] else "gagal (memakai cadangan)")
print(f"Waktu proses     : {hasil['waktu']:.3f} detik")

In [ ]:
# Tahap 1: dari foto asli sampai kertas lurus
tampilkan(
    [foto_contoh, hasil["tepi"], hasil["visual"], hasil["lurus"]],
    ["1. Foto asli", "2. Tepi Canny", "3. Sudut terdeteksi", "4. Perspektif lurus"],
    kolom=4, ukuran=4
)

In [ ]:
# Tahap 2: perbandingan tiga metode threshold
v = hasil["versi"]
tampilkan(
    [v["abu"], v["global"], v["otsu"], v["adaptif"]],
    ["Grayscale", "Threshold global (T=127)",
     f"Otsu (T={v['nilai_otsu']:.0f})", "Adaptive Gaussian"],
    kolom=4, ukuran=4
)

### Analisis Histogram (Sebelum dan Sesudah Equalization)
Histogram citra lurus dibandingkan dengan histogram setelah equalizeHist. Citra dikecilkan dulu agar nested loop tidak terlalu lama.

In [ ]:
abu_lurus = cv2.cvtColor(ubah_lebar(hasil["lurus"], 600), cv2.COLOR_BGR2GRAY)
abu_equal = cv2.equalizeHist(abu_lurus)

print("Histogram manual == cv2.calcHist (sebelum):", validasi_histogram(abu_lurus))
print("Histogram manual == cv2.calcHist (sesudah):", validasi_histogram(abu_equal))

hist_sebelum = histogram_manual(abu_lurus)
hist_sesudah = histogram_manual(abu_equal)

plt.figure(figsize=(14, 6))
for i, (gambar, hist, judul) in enumerate(
        [(abu_lurus, hist_sebelum, "Sebelum equalization"),
         (abu_equal, hist_sesudah, "Sesudah equalization")]):
    plt.subplot(2, 2, i + 1)
    plt.imshow(gambar, cmap="gray", vmin=0, vmax=255)
    plt.title(judul)
    plt.axis("off")
    plt.subplot(2, 2, i + 3)
    plt.bar(np.arange(256), hist, width=1.0)
    plt.xlabel("Intensitas")
    plt.ylabel("Jumlah piksel")
    plt.xlim(0, 255)
plt.tight_layout()
plt.show()

### Pengujian pada Seluruh Foto dan Evaluasi
Setiap foto diproses, lalu tiga ukuran dicatat:

* Berhasil 4 sudut: apakah sudut kertas ditemukan tanpa memakai cadangan.
* Rasio hasil: sisi panjang dibagi sisi pendek. Untuk kertas A4 nilainya mendekati 1,414.
* Selisih rasio A4 (%): seberapa jauh rasio hasil dari rasio A4. Hanya dihitung untuk foto dengan a4=True yang sudutnya terdeteksi benar.

Foto yang gagal dibaca dilewati dan dicatat, supaya satu URL yang salah tidak menghentikan seluruh pengujian.

In [ ]:
semua_hasil = {}
baris_evaluasi = []

for item in DAFTAR_FOTO:
    try:
        foto = baca_gambar_url(item["url"])
        h = pindai_dokumen(foto)
    except Exception as galat:
        print(f"[LEWAT] {item['nama']}: {galat}")
        continue

    semua_hasil[item["nama"]] = (foto, h)

    rasio, selisih = None, None
    if h["lurus"] is not None:
        tinggi, lebar = h["lurus"].shape[:2]
        rasio = max(tinggi, lebar) / min(tinggi, lebar)
        # Selisih rasio hanya bermakna jika 4 sudut terdeteksi dengan benar
        if item["a4"] and h["berhasil"]:
            selisih = abs(rasio - RASIO_A4) / RASIO_A4 * 100

    baris_evaluasi.append({
        "Foto": item["nama"],
        "Kondisi": item["kondisi"],
        "Berhasil 4 sudut": "Ya" if h["berhasil"] else "Tidak",
        "Rasio hasil": round(rasio, 3) if rasio else None,
        "Selisih rasio A4 (%)": round(selisih, 1) if selisih is not None else None,
        "Waktu (detik)": round(h["waktu"], 3) if "waktu" in h else None,
    })

tabel = pd.DataFrame(baris_evaluasi)
tabel

In [ ]:
# Ringkasan angka untuk laporan
jumlah = len(tabel)
sukses = (tabel["Berhasil 4 sudut"] == "Ya").sum()
print(f"Foto berhasil diproses : {jumlah}")
print(f"Sudut terdeteksi benar : {sukses} dari {jumlah} ({sukses / jumlah * 100:.0f}%)")
print(f"Rata-rata selisih rasio A4 (hanya yang berhasil): {tabel['Selisih rasio A4 (%)'].mean():.1f}%")
print(f"Rata-rata waktu proses : {tabel['Waktu (detik)'].mean():.3f} detik")

In [ ]:
# Galeri semua hasil: foto asli, deteksi sudut, dan hasil adaptive
gambar_galeri, judul_galeri = [], []
for nama, (foto, h) in semua_hasil.items():
    gambar_galeri.append(foto)
    judul_galeri.append(f"{nama}: asli")
    gambar_galeri.append(h["visual"] if "visual" in h else h["kecil"])
    judul_galeri.append("deteksi sudut")
    gambar_galeri.append(h["versi"]["adaptif"] if h["lurus"] is not None else h["tepi"])
    judul_galeri.append("hasil adaptive" if h["lurus"] is not None else "gagal")

tampilkan(gambar_galeri, judul_galeri, kolom=3, ukuran=4)

In [ ]:
# Perbandingan tiga metode threshold pada setiap foto
gambar_banding, judul_banding = [], []
for nama, (foto, h) in semua_hasil.items():
    if h["lurus"] is None:
        continue
    for kunci, label in [("global", "global"), ("otsu", "Otsu"), ("adaptif", "adaptive")]:
        gambar_banding.append(h["versi"][kunci])
        judul_banding.append(f"{nama}: {label}")

tampilkan(gambar_banding, judul_banding, kolom=3, ukuran=4)

## Mode B: Menyambung Dua Foto (ORB, Matching, Homografi)

Alur mengikuti Prak 5:
* ORB mencari keypoint dan descriptor pada kedua foto.
* Brute-Force matcher dengan jarak Hamming memadankan descriptor (kNN dan ratio test untuk membuang padanan yang meragukan).
* findHomography dengan RANSAC menghitung matriks transformasi dari titik yang cocok.
* Foto kanan ditransformasi ke bidang foto kiri, lalu keduanya digabung.
* Area hitam di tepi dibuang berdasarkan kontur.

Foto kiri harus bagian kiri dokumen, dan kedua foto sebaiknya tumpang tindih sekitar 30 sampai 40 persen.

Pada notebook ini satu foto lebar dibaca dari URL lalu dipotong di program menjadi dua bagian yang tumpang tindih sekitar 24 persen, sehingga tidak perlu mengunggah file potongan. Karena kedua bagian berasal dari satu foto, pergeseran antar bagian hanya berupa geseran lurus, sehingga ini pengujian yang lebih mudah daripada dua foto yang diambil terpisah. Jika punya dua foto asli, ganti foto_kiri dan foto_kanan dengan baca_gambar_url untuk masing-masing foto.

In [ ]:
def deteksi_orb(abu):
    # Mencari keypoint dan descriptor ORB
    orb = cv2.ORB_create(nfeatures=5000)
    return orb.detectAndCompute(abu, None)


def cocokkan_knn(desc_kanan, desc_kiri, rasio=0.75):
    # Memadankan descriptor dengan kNN (k=2) lalu menyaring dengan ratio test Lowe
    matcher = cv2.BFMatcher(cv2.NORM_HAMMING)
    pasangan = matcher.knnMatch(desc_kanan, desc_kiri, k=2)

    baik = []
    for p in pasangan:
        if len(p) == 2 and p[0].distance < rasio * p[1].distance:
            baik.append(p[0])
    return baik


def sambung_foto(kiri, kanan):
    # Menyambung foto kiri dan kanan
    # Mengembalikan hasil gabungan, keypoint, padanan, dan status RANSAC
    abu_kiri = cv2.cvtColor(kiri, cv2.COLOR_BGR2GRAY)
    abu_kanan = cv2.cvtColor(kanan, cv2.COLOR_BGR2GRAY)

    kp_kiri, desc_kiri = deteksi_orb(abu_kiri)
    kp_kanan, desc_kanan = deteksi_orb(abu_kanan)

    padanan = cocokkan_knn(desc_kanan, desc_kiri)
    if len(padanan) < 10:
        raise ValueError(f"Padanan terlalu sedikit ({len(padanan)}). Tambah area tumpang tindih.")

    # queryIdx menunjuk foto kanan, trainIdx menunjuk foto kiri
    titik_kanan = np.float32([kp_kanan[m.queryIdx].pt for m in padanan])
    titik_kiri = np.float32([kp_kiri[m.trainIdx].pt for m in padanan])

    H, status = cv2.findHomography(titik_kanan, titik_kiri, cv2.RANSAC, 4.0)
    if H is None:
        raise ValueError("Homografi gagal dihitung.")

    # Hitung ukuran kanvas agar seluruh foto kanan yang tertransformasi muat
    tinggi_k, lebar_k = kiri.shape[:2]
    tinggi_r, lebar_r = kanan.shape[:2]
    sudut_kanan = np.float32([[0, 0], [lebar_r, 0], [lebar_r, tinggi_r], [0, tinggi_r]]).reshape(-1, 1, 2)
    sudut_kiri = np.float32([[0, 0], [lebar_k, 0], [lebar_k, tinggi_k], [0, tinggi_k]]).reshape(-1, 1, 2)
    sudut_kanan_baru = cv2.perspectiveTransform(sudut_kanan, H)

    semua_sudut = np.concatenate((sudut_kiri, sudut_kanan_baru))
    x_min, y_min = np.int32(semua_sudut.min(axis=0).ravel() - 0.5)
    x_max, y_max = np.int32(semua_sudut.max(axis=0).ravel() + 0.5)

    if (x_max - x_min) > 12000 or (y_max - y_min) > 12000:
        raise ValueError("Hasil homografi tidak masuk akal (kanvas terlalu besar).")

    # Geser agar tidak ada koordinat negatif
    geser = np.array([[1, 0, -x_min], [0, 1, -y_min], [0, 0, 1]], dtype=np.float64)
    gabungan = cv2.warpPerspective(kanan, geser @ H, (x_max - x_min, y_max - y_min))
    gabungan[-y_min:tinggi_k - y_min, -x_min:lebar_k - x_min] = kiri

    return gabungan, kp_kiri, kp_kanan, padanan, int(status.sum())


def potong_area_hitam(gambar):
    # Membuang tepi hitam hasil warp dengan bounding box kontur terbesar (seperti Prak 5)
    abu = cv2.cvtColor(gambar, cv2.COLOR_BGR2GRAY)
    _, biner = cv2.threshold(abu, 0, 255, cv2.THRESH_BINARY)
    kontur, _ = cv2.findContours(biner, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    terbesar = max(kontur, key=cv2.contourArea)
    x, y, w, h = cv2.boundingRect(terbesar)
    return gambar[y:y + h, x:x + w]

In [ ]:
def potong_dua(foto, batas_kiri=0.62, batas_kanan=0.38):
    # Memotong satu foto lebar menjadi bagian kiri dan kanan yang saling tumpang tindih
    lebar = foto.shape[1]
    return foto[:, :int(lebar * batas_kiri)], foto[:, int(lebar * batas_kanan):]

In [ ]:
# Menjalankan Mode B pada foto lebar pertama
item = FOTO_LEBAR[0]
foto_lebar = ubah_lebar(baca_gambar_url(item["url"]), 2000)
foto_kiri, foto_kanan = potong_dua(foto_lebar, item["batas_kiri"], item["batas_kanan"])

tampilkan([foto_lebar, foto_kiri, foto_kanan],
          ["Foto lebar (asli)", "Bagian kiri", "Bagian kanan"], kolom=3, ukuran=6)

gabungan, kp_kiri, kp_kanan, padanan, jumlah_inlier = sambung_foto(foto_kiri, foto_kanan)
print("Keypoint kiri / kanan :", len(kp_kiri), "/", len(kp_kanan))
print("Padanan lolos ratio test :", len(padanan))
print("Padanan lolos RANSAC (inlier):", jumlah_inlier)

# Gambar garis padanan terbaik antara kedua foto
terbaik = sorted(padanan, key=lambda m: m.distance)[:50]
garis = cv2.drawMatches(foto_kanan, kp_kanan, foto_kiri, kp_kiri, terbaik, None,
                        flags=cv2.DrawMatchesFlags_NOT_DRAW_SINGLE_POINTS)
tampilkan([garis], ["50 padanan terbaik (kanan di kiri, kiri di kanan)"], kolom=1, ukuran=14)

In [ ]:
# Hasil sambungan, dipotong, lalu diberi tahap percantik yang sama dengan Mode A
sambungan = potong_area_hitam(gabungan)
versi_b = percantik(sambungan)

tampilkan(
    [gabungan, sambungan, versi_b["adaptif"]],
    ["Hasil gabungan mentah", "Setelah dipotong", "Hasil adaptive"],
    kolom=3, ukuran=7
)

### Kesimpulan Singkat
Isi bagian ini setelah seluruh cell dijalankan dan angka evaluasi pada Bagian 8 sudah muncul.

* Tingkat keberhasilan deteksi 4 sudut: (isi dari ringkasan angka)
* Metode threshold terbaik pada foto berbayangan: (isi hasil pengamatan)
* Kondisi foto yang membuat deteksi gagal: (isi, misalnya kertas putih di meja putih)
* Kualitas sambungan Mode B: (isi jumlah inlier dan penilaian visual)
* Saran perbaikan: (isi untuk Bab 5 laporan)